# **Uber Ride Demand Analysis: Temporal and Spatial Insights**

This analysis delves into Uber ride data from New York City, uncovering patterns in ride demand, weather impacts, and competition with yellow and green taxis. Leveraging feature engineering, regression models, and clustering techniques, the project provides actionable insights to enhance operational efficiency.

**Dataset Overview**

The dataset includes ride data for Uber, yellow cabs, and green cabs in NYC, complemented by weather information. Key attributes include:
- Date/Time Features: Ride timestamps, aggregated hourly and daily, to analyze temporal patterns.
- Location Features: Pickup coordinates and borough-level trends for spatial insights.
- Weather Data: Precipitation levels as an external factor influencing demand.

**Objective**

The objective is to identify temporal and spatial demand trends and evaluate the influence of external factors such as weather, providing insights into ride-sharing and taxi operations.


---

#**1. Imports**

In [ ]:
# Core libraries for system operations and time handling
import os
import time
from datetime import datetime

# Data manipulation and analysis
import numpy as np
import pandas as pd

# Data visualization
import seaborn as sns
import matplotlib.pyplot as plt
# plt.style.use('ggplot')  # Optional: Uncomment for consistent ggplot-style plots

# Machine learning and statistical modeling
import sklearn  # Scikit-learn for ML tasks
import statsmodels.api as sm  # For statistical modeling

# Deep learning
import tensorflow as tf
# tf.keras.utils.set_random_seed(42)  # Optional: Uncomment to ensure reproducibility

# Additional setup: Stargazer for summarizing regression results
!pip install stargazer  # Install Stargazer for formatted regression summaries

# Google Colab-specific setup: Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

# Define data path (update path as needed for your dataset)
data_path = '/content/drive/yourpath'

# Verify the current working directory
os.getcwd()


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


'/content'

      Temporal Feature Engineering:
      Extracted features such as `Hour`, `DayOfWeek`, and `Month` from timestamps to study time-based demand trends.

In [ ]:
# Utility function to add date and time features
def add_date_features(df, datetime_column):
    """
    Adds date and time features to a DataFrame based on a datetime column.

    Parameters:
        df (DataFrame): The input DataFrame.
        datetime_column (str): The column name containing datetime values.

    Returns:
        DataFrame: The modified DataFrame with new date and time features.
    """
    # Convert column to datetime format
    df[datetime_column] = pd.to_datetime(df[datetime_column])

    # Extract date and time features
    df['Date'] = df[datetime_column].dt.date
    df['Month'] = df[datetime_column].dt.month
    df['Week'] = df[datetime_column].dt.isocalendar().week
    df['DayOfMonthNum'] = df[datetime_column].dt.day
    df['DayOfWeekNum'] = df[datetime_column].dt.dayofweek
    df['DayOfWeek'] = df[datetime_column].dt.day_name()
    df['Hour'] = df[datetime_column].dt.hour

    return df


In [ ]:
# Load aggregated data for analysis
num_rides_rainfall_June_2015 = pd.read_csv(os.path.join(data_path, 'demand_elasticity_data_NYC_June_2015.csv'))
num_rides_rainfall_June_2015['Date'] = pd.to_datetime(num_rides_rainfall_June_2015['Date'])
num_rides_rainfall_June_2015.set_index(['Date', 'Hour', 'LocationID'], inplace=True)
display(num_rides_rainfall_June_2015)

Num_Uber_Rides  Num_Yellow_Cab_Rides  \
Date       Hour LocationID                                         
2015-06-01 0    7                      8.0                  48.0   
                17                    17.0                  14.0   
                18                     2.0                   2.0   
                25                     7.0                  11.0   
                28                     1.0                   3.0   
...                                    ...                   ...   
2015-06-30 22   255                   88.0                  87.0   
                256                   67.0                  59.0   
                257                    4.0                   1.0   
                260                    8.0                   4.0   
                263                   36.0                 317.0   

                            Num_Green_Cab_Rides  precipitation  
Date       Hour LocationID                                      
2015-06-01 0    7                          92.0      69.415139  
                17                         32.0      47.720132  
                18                          2.0      41.290715  
                25                         35.0      19.821492  
                28                          1.0      61.365975  
...                                         ...            ...  
2015-06-30 22   255                       133.0      50.401150  
                256                        58.0      51.586199  
                257                         2.0      47.245371  
                260                        42.0      72.664295  
                263                         2.0      35.342761  

[47218 rows x 4 columns]

---

# **2.Regression Analysis: Uber vs. Green and Yellow Taxis**

      Objective:
      - To analyze the relationship between Uber ride counts and taxi ride counts (green and yellow taxis).

      Model Used:
      - Ordinary Least Squares (OLS) regression.



In [ ]:
# Simple Regression: Yellow Taxi vs Uber Rides
reg = sm.OLS(num_rides_rainfall_June_2015['Num_Yellow_Cab_Rides'],
             sm.add_constant(num_rides_rainfall_June_2015['Num_Uber_Rides']))
results = reg.fit()
print(results.summary())

# Simple Regression: Green Taxi vs Uber Rides
reg = sm.OLS(num_rides_rainfall_June_2015['Num_Green_Cab_Rides'],
             sm.add_constant(num_rides_rainfall_June_2015['Num_Uber_Rides']))
results = reg.fit()
print(results.summary())

# Combined Taxi Demand vs Uber Rides
combined_taxi_rides = num_rides_rainfall_June_2015[['Num_Yellow_Cab_Rides', 'Num_Green_Cab_Rides']].sum(axis=1)
reg = sm.OLS(combined_taxi_rides, sm.add_constant(num_rides_rainfall_June_2015['Num_Uber_Rides']))
results = reg.fit()
print(results.summary())


                             OLS Regression Results                             
Dep. Variable:     Num_Yellow_Cab_Rides   R-squared:                       0.363
Model:                              OLS   Adj. R-squared:                  0.363
Method:                   Least Squares   F-statistic:                 2.695e+04
Date:                  Fri, 13 Dec 2024   Prob (F-statistic):               0.00
Time:                          11:03:30   Log-Likelihood:            -2.7904e+05
No. Observations:                 47218   AIC:                         5.581e+05
Df Residuals:                     47216   BIC:                         5.581e+05
Df Model:                             1                                         
Covariance Type:              nonrobust                                         
                     coef    std err          t      P>|t|      [0.025      0.975]
----------------------------------------------------------------------------------
const            -16.078

    Regression Insights: Uber, Yellow Taxis, and Green Taxis

    Key Findings:

    1. Yellow Taxi Rides vs. Uber Rides
    - Positive Correlation: The coefficient for `Num_Uber_Rides` is 3.2594, indicating a strong positive relationship. For every additional Uber ride, the number of yellow taxi rides increases by approximately 3.26.
    - Implications: Suggests that Uber and yellow taxis might cater to overlapping high-demand areas, leading to complementary ride increases during peak periods.
    - Significance: The p-value (< 0.05) confirms this relationship is statistically significant.

    2. Green Taxi Rides vs. Uber Rides
    - Positive Correlation: The coefficient for `Num_Uber_Rides` is 0.7762, indicating a weaker positive relationship compared to yellow taxis. For every additional Uber ride, green taxi rides increase by approximately 0.78.
    - Implications: Green taxis primarily operate in outer boroughs or underserved areas, which overlap less with Uber's core market compared to yellow taxis.
    - Significance: The p-value (< 0.05) confirms statistical significance.

    3. Combined Demand Impact
    - The combined analysis shows a coefficient of 4.0357 for `Num_Uber_Rides`, indicating that total taxi demand (yellow + green) correlates strongly with Uber rides.
    - Implications: This highlights that Uber does not completely replace traditional taxi services; instead, it competes while potentially complementing demand during peak times or in specific areas.


---

# **3. Regression Analysis: Uber vs. Green and Yellow Taxis (Controlling for Rainfall)**

    Objective
    - To analyze the relationship between Uber ride counts and taxi ride counts (yellow and green taxis) while accounting for precipitation as a potential confounding variable.

    Model Used
    - Ordinary Least Squares (OLS) regression, with precipitation included as a control variable to assess its impact on the relationship between Uber rides and taxi rides.

In [ ]:
#### Control for Rainfall

# Regression controlling for precipitation
reg = sm.OLS(num_rides_rainfall_June_2015['Num_Yellow_Cab_Rides'],
             sm.add_constant(num_rides_rainfall_June_2015[['Num_Uber_Rides', 'precipitation']]))
results = reg.fit()
print(results.summary())

reg = sm.OLS(num_rides_rainfall_June_2015['Num_Green_Cab_Rides'],
             sm.add_constant(num_rides_rainfall_June_2015[['Num_Uber_Rides', 'precipitation']]))
results = reg.fit()
print(results.summary())

reg = sm.OLS(num_rides_rainfall_June_2015[['Num_Yellow_Cab_Rides', 'Num_Green_Cab_Rides']].sum(axis=1),
             sm.add_constant(num_rides_rainfall_June_2015[['Num_Uber_Rides', 'precipitation']]))
results = reg.fit()
print(results.summary())

                             OLS Regression Results                             
Dep. Variable:     Num_Yellow_Cab_Rides   R-squared:                       0.363
Model:                              OLS   Adj. R-squared:                  0.363
Method:                   Least Squares   F-statistic:                 1.348e+04
Date:                  Fri, 13 Dec 2024   Prob (F-statistic):               0.00
Time:                          11:04:12   Log-Likelihood:            -2.7904e+05
No. Observations:                 47218   AIC:                         5.581e+05
Df Residuals:                     47215   BIC:                         5.581e+05
Df Model:                             2                                         
Covariance Type:              nonrobust                                         
                     coef    std err          t      P>|t|      [0.025      0.975]
----------------------------------------------------------------------------------
const            -16.836

    Key Insights

    1. Yellow Taxi Rides
    - Uber Rides: Positive correlation (coefficient = 3.2596, p-value < 0.05), showing a strong association between Uber rides and yellow taxi rides, regardless of weather conditions.
    - Precipitation: Minimal impact (coefficient = 0.0226, p-value = 0.414), suggesting that rainfall does not significantly influence yellow taxi demand.

    2. Green Taxi Rides
    - Uber Rides: Positive correlation (coefficient = 0.7762, p-value < 0.05), indicating that Uber rides weakly correlate with green taxi rides.
    - Precipitation: Insignificant (coefficient = 0.0017, p-value = 0.884), implying negligible weather-related effects on green taxi demand.

    3. Combined Taxi Demand (Yellow + Green)
    - Uber Rides: Strong positive correlation (coefficient = 4.0359, p-value < 0.05), reflecting the combined influence of Uber rides on total taxi demand.
    - Precipitation: Negligible impact (coefficient = 0.0244, p-value = 0.366), indicating that rainfall does not directly influence combined taxi demand.

    Summary:

    Controlling for precipitation allows us to confirm that the positive correlation between Uber rides and taxi rides is not significantly influenced by weather. This analysis demonstrates that demand for Uber and taxis largely depends on overlapping geographic and temporal factors rather than weather conditions.


---

# **4. Regression Analysis: Temporal and Spatial Fixed Effects**

    Objective
    - Investigate the effect of Uber rides on yellow cab rides using temporal (hourly) and spatial (location-based) fixed effects.
    - Control for precipitation to ensure robust estimates of the relationship.

In [ ]:
#### Generate Dummy Variables

# Add temporal fixed effects using dummy variables
hour_fixed_effects = pd.get_dummies(num_rides_rainfall_June_2015.reset_index(), columns=['Hour'], drop_first=True).astype(int)

# Regression with hourly fixed effects
reg_hour = sm.OLS(hour_fixed_effects['Num_Yellow_Cab_Rides'],
                  sm.add_constant(hour_fixed_effects.drop(columns=['Num_Yellow_Cab_Rides', 'Date', 'LocationID'])))
results_hour = reg_hour.fit()
print(results_hour.summary())

# Add spatial fixed effects using dummy variables
location_fixed_effects = pd.get_dummies(num_rides_rainfall_June_2015.reset_index(), columns=['LocationID'], drop_first=True).astype(int)

# Regression with spatial fixed effects
reg_location = sm.OLS(location_fixed_effects['Num_Yellow_Cab_Rides'],
                      sm.add_constant(location_fixed_effects.drop(columns=['Num_Yellow_Cab_Rides', 'Date', 'Hour'])))
results_location = reg_location.fit()
print(results_location.summary())

# Add both temporal and spatial fixed effects
temp_spatial_fixed_effects = pd.get_dummies(num_rides_rainfall_June_2015.reset_index(),
                                            columns=['Hour', 'LocationID'], drop_first=True).astype(int)

reg_temp_spatial = sm.OLS(temp_spatial_fixed_effects['Num_Yellow_Cab_Rides'],
                          sm.add_constant(temp_spatial_fixed_effects.drop(columns=['Num_Yellow_Cab_Rides', 'Date'])))
results_temp_spatial = reg_temp_spatial.fit()
print(results_temp_spatial.summary())


                             OLS Regression Results                             
Dep. Variable:     Num_Yellow_Cab_Rides   R-squared:                       0.417
Model:                              OLS   Adj. R-squared:                  0.417
Method:                   Least Squares   F-statistic:                     1297.
Date:                  Fri, 13 Dec 2024   Prob (F-statistic):               0.00
Time:                          11:13:16   Log-Likelihood:            -2.7697e+05
No. Observations:                 47218   AIC:                         5.540e+05
Df Residuals:                     47191   BIC:                         5.542e+05
Df Model:                            26                                         
Covariance Type:              nonrobust                                         
                          coef    std err          t      P>|t|      [0.025      0.975]
---------------------------------------------------------------------------------------
const         

      Key Insights
      1. Hourly Fixed Effects:
        - Certain hours (e.g., 1 a.m. to 4 a.m. and late afternoons) show significantly higher yellow cab demand.
        - Temporal patterns highlight clear peaks in demand during specific hours of the day.

      2. Spatial Fixed Effects:
        - High-demand zones such as airports and business districts exhibit stronger relationships with Uber ride counts.
        - Some areas show negligible or non-significant effects on yellow cab demand.

      3. Combined Temporal and Spatial Effects:
        - R-squared increases significantly with combined fixed effects (0.857), explaining a large proportion of yellow cab ride variance.
        - Hourly and spatial coefficients highlight both time- and location-specific demand trends.

---

#**5. Conclusion**

- **Temporal Patterns**: Hourly fixed effects reveal significant peaks in yellow cab demand during early mornings and late afternoons, aligning with typical commuter and nightlife activities.
- **Spatial Trends**: Location-based fixed effects highlight demand hotspots, such as airports and business districts, confirming their role as key zones for taxi services.
- **Combined Effects**: Incorporating both temporal and spatial fixed effects significantly improves the model's explanatory power (R-squared ~0.857), showcasing the importance of accounting for both time and location in demand analysis.
- **Precipitation Impact**: Rainfall has minimal influence on yellow cab and Uber ride demand, suggesting that weather conditions alone are not strong determinants of ride distribution.
- **Shared Dynamics**: A strong positive relationship between Uber and yellow cab rides highlights shared demand patterns, indicating complementary or substitutive effects in various contexts.

  These findings can guide resource allocation strategies, such as dynamic driver positioning, and inform policy decisions on urban transportation systems.
